```mermaid
flowchart LR
    A0["00"] --> A1a["01a"] --> A1b["01b"] --> A2["02"] --> A3["03"] --> A4a["04a"] --> A4b["04b"]
    A4b --> A5a["05a"] --> A5b["05b"] --> A6a["06a"] --> A6b["06b"]
    A6b --> A7["07"] --> A8a["08a"] --> A8b["08b"]
    A8b --> A9["09"] --> A10["10"] --> A11["11"] --> A12["12"] 
    
    classDef normal fill:#f8f9fa,stroke:#adb5bd,stroke-width:1px,color:#111;
    classDef done fill:#e8f7f0,stroke:#198754,stroke-width:1.5px,color:#111;
    classDef current fill:#fff3cd,stroke:#ff8c00,stroke-width:2px,color:#111;
    
    class A0,A1a,A1b,A2,A3,A4a,A4b,A5a,A5b,A6a,A6b done;
    class A7 current;
    class A8a,A8b,A9,A10,A11,A12 normal;
```

# Notebook 07 — Text Representation for Classification: Counts, TF–IDF, and Feature Design

This notebook introduces classical text representations for supervised learning and exploratory classification tasks. We move from descriptive exploration toward *features* that can be used as inputs for machine-learning models.

The central question is: how do we transform text into numerical representations that preserve useful information about meaning, style, concepts, or historical context?

We focus on three families of representations:
- Bag-of-Words (BoW) counts
- n-grams (multi-word sequences)
- TF–IDF (weighted lexical features)

Throughout the notebook, we connect representations back to the course theme: knowledge dynamics and conceptual change across philosophical texts over time.

**Important methodological point:** representations are not neutral. Every preprocessing decision (tokenization, stopword removal, n-grams, weighting) changes what the model can learn and what patterns become visible.

## Learning goals

By the end of this notebook, students should be able to:

- understand the assumptions behind Bag-of-Words representations
- explain the difference between counts and TF–IDF
- create reusable sparse feature matrices
- inspect feature vocabularies and top-weighted terms
- compare representations across time bins
- prepare reusable features for downstream classification tasks
- reflect on what lexical representations capture — and what they miss

## Method note

This notebook deliberately focuses on interpretable classical representations before moving to classification. TF–IDF and count vectors remain extremely useful in NLP because:

- they are fast and CPU-friendly
- they are transparent and interpretable
- they provide strong baselines
- they reveal what lexical signals models are actually using

Embeddings will later provide complementary semantic representations, but classical lexical features are still foundational for understanding NLP workflows.

In [ ]:
from __future__ import annotations

from pathlib import Path
import re
import json
import math
from collections import Counter
from tqdm.auto import tqdm

import numpy as np
import pandas as pd

from scipy import sparse

from sklearn.feature_extraction.text import (
    CountVectorizer,
    TfidfVectorizer,
    ENGLISH_STOP_WORDS,
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

import spacy
from spacy.tokens import DocBin

In [ ]:
# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
PROJECT_ROOT = Path('.')

DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'

TEXTS_DIR = PROCESSED_DIR / 'cleaned'

ANALYSIS_DIR = PROJECT_ROOT / 'analysis'
FIGURES_DIR = ANALYSIS_DIR / 'figures'
TABLES_DIR = ANALYSIS_DIR / 'tables'
REPORTS_DIR = ANALYSIS_DIR / 'reports'
MODELS_DIR = ANALYSIS_DIR / 'models'

CACHE_DIR = PROJECT_ROOT / 'cache'

# Canonical document index
DOC_INDEX = TABLES_DIR / 'nb03-doc_index.csv'

# Full metadata
METADATA = TABLES_DIR / 'nb02-corpus-complete-metadata.csv'

# Split DocBin files from Notebook 05a
SPLIT_DIR = PROCESSED_DIR / 'nb05-corpus-split'

# Plot settings
MAX_DOCS_PLOT = 300

print('DOC_INDEX:', DOC_INDEX)
print('SPLIT_DIR:', SPLIT_DIR)
print('METADATA:', METADATA)

## Loading the canonical document table

We load the canonical document index created earlier in the workflow. This table acts as the stable reference for all later notebooks: document IDs, filenames, time bins, and metadata are kept consistent across sessions.

In [ ]:
df = pd.read_csv(DOC_INDEX)
meta = pd.read_csv(METADATA)

print('Documents:', len(df))
display(df.head())

print('-' * 80)
print('Metadata:')
display(meta.head(1))

## Loading the spaCy corpus from Notebook 05a

Notebook 05a serialized the corpus into split spaCy `DocBin` files. This allows us to reload annotated documents without reparsing the entire corpus.

Why does this matter?
- linguistic annotation is computationally expensive
- serialized artifacts make workflows reproducible
- later notebooks can focus on analysis rather than preprocessing

In [ ]:
# Get all .spacy files
spacy_files = sorted(SPLIT_DIR.glob('*.spacy'))
print(f'Found {len(spacy_files)} split files.')

# Load spacy's lightweight pipeline vocabulary and disable the NER pipeline
nlp = spacy.load('en_core_web_sm', disable=['ner'])

docs = []
for fp in spacy_files:
    db = DocBin().from_disk(fp)
    docs.extend(list(db.get_docs(nlp.vocab)))

print('Loaded docs:', len(docs))

## Reconstructing document texts

For vectorization we need raw text strings. We reconstruct them from the spaCy documents and align them with the canonical document table.

In [ ]:
chunk_rows = []
for doc in docs:
    # Retrieve metadata from the Doc
    pg_id = doc.user_data.get('pg_id')
    title = doc.user_data.get('title')
    publication_year = doc.user_data.get('publication_year')
    time_bin = doc.user_data.get('time_bin')
    chunk_index = doc.user_data.get('chunk_index', 0)
    
    chunk_rows.append({
        'pg_id': pg_id,
        'title': title,
        'publication_year': publication_year,
        'time_bin': time_bin,
        'chunk_index': chunk_index,
        'text': doc.text,          # the actual text of this chunk
        'n_tokens': len(doc)
    })

chunk_df = pd.DataFrame(chunk_rows)
chunk_df["publication_year"] = chunk_df["publication_year"].astype("Int64")
print(f"\nCreated chunk DataFrame with {len(chunk_df)} rows.")
display(chunk_df.head())

## Tokenization choices

Classical vectorizers require tokenized text.

Important methodological questions:
- Should punctuation be removed?
- Should stopwords be removed?
- Should we lowercase?
- Should we keep numbers?
- Should we keep very short tokens?

There is no universally correct answer. The best choice depends on the research question.

# Stop words

# Fill the gap

### _Store sklearn English stop words in the given variable_ `STOP`

In [ ]:
# =============================================== YOUR CODE HERE ===============================================
# We loaded sklearn English stopwords in the configuration import
# Find the list and store it as a set in STOP
STOP = 

# Fill the gap

### _How many stop words in STOP?_

In [ ]:
# =============================================== YOUR CODE HERE ===============================================
# How many stopwords does STOP contain?
nb_stopwords =
print(f"\nNumber of English stopwords in sklearn list: {nb_stopwords}.")

## Random Chunks

Let’s look at 3 random chunks from the corpus.

In [ ]:
# Get a random sample of chunks
sample_chunks = chunk_df.sample(n=3, random_state=None)  # random_state=None means different each run

for idx, row in sample_chunks.iterrows():
    print(f"\n{'='*60}")
    print(f"PG ID: {row['pg_id']}\n")
    print(f"Title: {row['title']} (time bin: {row['time_bin']})\n")
    print(f"Chunk size: {len(row['text'])} characters, {len(row['text'].split())} tokens")
    print(f"Text snippet (first 500 chars):\n{row['text'][:500]}...")
    print()

# Fill the gap 

### _Write a function that analyses stopword impact_
 
Complete the function below to compute:
 - Total number of tokens in a text.
 - Number of tokens that are stopwords (using a given set).
 - Percentage of stopwords.
 - Top 10 most frequent stopwords.

In [ ]:
def stopword_summary(text:str, stop_set:set, top_nb=10) -> dict:
    """
    Given a text string and a set of stopwords, return a dictionary with:
      - total_tokens: int
      - stopword_tokens: int
      - stopword_percentage: float
      - top_stopwords: list of (word, count) tuples (top 10)
    """

# =============================================== YOUR CODE HERE ===============================================
    tokens =               # Hint: split the text into lowercased tokens (whitespace split) using string methods

# =============================================== YOUR CODE HERE ===============================================
    # Store all the stopwords tokens which can be found in stop_set
    stop_tokens =          # Hint: use a list comprehension

    # Remove punctuation
    tokens = [re.sub(r'[^\w\s]', '', t) for t in tokens]
    
    # Remove empty strings
    tokens = [t for t in tokens if t] 

# =============================================== YOUR CODE HERE ===============================================
    # Store in the variable `total` the total number of tokens
    total = 
    
# =============================================== YOUR CODE HERE ===============================================
    # Store in the variable `stop_count` the total number of tokens that are stopwords
    stop_count = 

    # Percentage of stopwords
    pct = (stop_count / total * 100) if total > 0 else 0
    
    counter = Counter(stop_tokens)
    top = counter.most_common(top_nb)
    
    return {
        'total_tokens': total,
        'stopword_tokens': stop_count,
        'stopword_percentage': pct,
        'top_stopwords': top
    }

In [ ]:
# Test stopword_summary on a sample chunk
# If the funtion is right, the code below should run seamlessly
sample_text = chunk_df.sample(70).iloc[0]['text']
result = stopword_summary(sample_text, STOP, 20)
print("\nSample result:\n")
print(f"Total tokens: {result['total_tokens']}")
print(f"Stopwords: {result['stopword_tokens']} ({result['stopword_percentage']:.1f}%)")
print("Top stopwords:", result['top_stopwords'])

# Fill the gap

Use the code from notebook 04a to update the default stop word list.

In [ ]:
STOP_WORDS_FILE = Path('./analysis/stop_words_custom.txt')

def load_stopwords(filepath:Path = STOP_WORDS_FILE) -> set:
# =============================================== YOUR CODE HERE ===============================================

    return stopwords

CUSTOM_STOPWORDS = load_stopwords()
print(f"Loaded {len(CUSTOM_STOPWORDS)} custom stop words.")

# Fill the gap

Merge the custom stop word list with the default one as we did before without repetitions.

In [ ]:
# =============================================== YOUR CODE HERE ===============================================
STOPWORDS = 

# Fill the gap

### _Measure the impact of the custom stopwords list_

Use the code above with which we tested our function `stopword_summary` on a sample chunk to test the same chunk with our custom `STOPWORDS` instead of the default list.

In [ ]:
# Analyse the same chunks as above with your custom list
print("Custom list of stop words analysis:\n")
# =============================================== YOUR CODE HERE ===============================================

## Stopword Comparison: Raw vs Default vs Custom

This cell compares the effect of different stopword treatments on a set of random chunks.
For each chunk, it shows:
- Total number of unique tokens
- Top 10 most frequent tokens (with and without stopwords)
- Overlap in top tokens between Raw and Custom

**Task**: Run this cell after you have defined your `STOPWORDS` set. Discuss the differences.

In [ ]:
# ------------------------------------------------------------------------------
# Helper: tokenize and count
# ------------------------------------------------------------------------------
def token_counter(text, stop_set=None):
    """
    Tokenize a text (lowercase, remove punctuation) and return a Counter.
    If stop_set is provided, exclude tokens in that set.
    """
    # Lowercase and split
    tokens = text.lower().split()
    # Remove punctuation (simple regex)
    tokens = [re.sub(r'[^\w\s]', '', t) for t in tokens]
    tokens = [t for t in tokens if t]  # remove empty strings
    
    if stop_set is not None:
        tokens = [t for t in tokens if t not in stop_set]
    
    return Counter(tokens)

In [ ]:
# ------------------------------------------------------------------------------
# Select a few random chunks (same seeds for reproducibility)
# ------------------------------------------------------------------------------
# Use a fixed seed so everyone sees the same chunks, or random_state=None for different.
sample_chunks = chunk_df.sample(n=3, random_state=41)  # change seed for variation

TOP_N = 20

In [ ]:
# ------------------------------------------------------------------------------
# Analyse each chunk
# ------------------------------------------------------------------------------
comparison_rows = []

for idx, row in sample_chunks.iterrows():
    text = row['text']
    
    # Counts
    raw_counter = token_counter(text, stop_set=None)
    default_counter = token_counter(text, stop_set=STOP)
    custom_counter = token_counter(text, stop_set=CUSTOM_STOPWORDS)
    
    # Summary stats
    comparison_rows.append({
        'pg_id': row['pg_id'],
        'title': row['title'],
        'raw_unique': len(raw_counter),
        'default_unique': len(default_counter),
        'custom_unique': len(custom_counter),
        'raw_top': raw_counter.most_common(TOP_N),
        'default_top': default_counter.most_common(TOP_N),
        'custom_top': custom_counter.most_common(TOP_N),
    })

In [ ]:
# ------------------------------------------------------------------------------
# Display comparison table
# ------------------------------------------------------------------------------
print("=" * 80)
print("STOPWORD COMPARISON: Raw vs Default vs Custom")
print("=" * 80)

for row in comparison_rows:
    print(f"\n📄 PG ID: {row['pg_id']}  |  {row['title'][:50]}...")
    print(f"   Unique tokens (raw):     {row['raw_unique']}")
    print(f"   Unique tokens (default): {row['default_unique']} (removed {row['raw_unique'] - row['default_unique']})")
    print(f"   Unique tokens (custom):  {row['custom_unique']} (removed {row['raw_unique'] - row['custom_unique']})")
    
    print(f"\n   Top {TOP_N} Raw:")
    for word, count in row['raw_top']:
        print(f"      {word}: {count}")
    
    print(f"\n   Top {TOP_N} Default:")
    for word, count in row['default_top']:
        print(f"      {word}: {count}")
    
    print(f"\n   Top {TOP_N} Custom:")
    for word, count in row['custom_top']:
        print(f"      {word}: {count}")
    
    print("\n" + "-" * 60)


In [ ]:
# ------------------------------------------------------------------------------
# Bar plot: Compare TOP_N tokens for raw vs custom on the first chunk
# ------------------------------------------------------------------------------
# Pick the first chunk for a visual comparison
first = comparison_rows[0]
raw_top = dict(first['raw_top'])
custom_top = dict(first['custom_top'])

# Get union of top words (from both)
all_words = set(raw_top.keys()) | set(custom_top.keys())

# Build DataFrame for plotting
plot_df = pd.DataFrame({
    'word': list(all_words),
    'raw_count': [raw_top.get(w, 0) for w in all_words],
    'custom_count': [custom_top.get(w, 0) for w in all_words]
}).sort_values('raw_count', ascending=False).head(TOP_N)

# Plot using pandas bar
ax = plot_df.plot(
    kind='bar', 
    x='word', 
    y=['raw_count', 'custom_count'],
    figsize=(10, 6),
    color=['teal', 'gold'],
    legend=True
)
ax.set_title(f"Top {TOP_N} words: raw vs custom list of stop words\n(PG ID: {first['pg_id']})")
ax.set_xlabel('Word')
ax.set_ylabel('Frequency')
ax.legend(['Raw', 'Custom'])
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
MIN_TOKEN_LEN = 2

def tokenize(text: str) -> list[str]:
    toks = []
    for t in nlp.make_doc(str(text).lower()):
        # Skip spaces and punctuation (spaCy's token-level check)
        if t.is_space or t.is_punct:
            continue
        # Skip possessive markers and particles
        if t.tag_ in ['POS', 'PART']:
            continue
        
        # Strip leading/trailing punctuation from the token text
        tok = re.sub(r'^[^\w]+|[^\w]+$', '', t.text)
        
        # Skip if empty after stripping
        if not tok:
            continue
        
        # Skip if contains a digit
        if re.search(r'\d', tok):
            continue
        
        # Skip if too short
        if len(tok) < MIN_TOKEN_LEN:
            continue
        
        # Skip if in stopwords
        if tok in STOPWORDS: # or STOP
            continue
        
        # Must contain at least one alphabetic character
        if not re.search(r'[a-z]', tok):
            continue

        # Skip tokens that are roman numerals (e.g., i, ii, iii, iv, v, vi, vii, viii, ix, x)
        roman_pattern = r'^[ivxlcdm]+$'
        if re.match(roman_pattern, tok):
            continue
        
        toks.append(tok)
    
    return toks

In [ ]:
print(tokenize(chunk_df.iloc[0]['text'])[:30])

# Bag-of-Words (BoW) representations

Bag-of-Words models represent documents as frequency vectors.

Main assumption:
- a document can be represented by which words it contains and how often they occur

What BoW ignores:
- syntax
- word order (mostly)
- semantic similarity
- discourse structure

Despite these limitations, BoW often performs surprisingly well in text classification tasks.

In [ ]:
# ------------------------------------------------------------
# Vectorization settings
# ------------------------------------------------------------
# These control the size and shape of the vocabulary before we build
# any vectors. Tuning these is a real design decision: too permissive
# and you get a huge, noisy vocabulary; too strict and you lose
# meaningful (often rare but important) terms.

MIN_DF = 3
# Ignore terms that appear in fewer than MIN_DF documents (chunks).
# This filters out typos, OCR noise, and one-off words that don't
# generalize — but setting it too high can drop rare, meaningful terms.

MAX_FEATURES = 50000
# Cap the vocabulary at the MAX_FEATURES most frequent terms (by
# corpus-wide count). Keeps the matrix a manageable size; anything
# beyond this cutoff is simply excluded from the vocabulary.

NGRAM_RANGE_BOW = (1, 2)
# Unigrams + bigrams. This is our baseline "bag-of-words" model —
# it still counts single words, but also short two-word phrases
# ("natural law", "free will") that a pure unigram model would miss.

# ------------------------------------------------------------
# Model setting
# ------------------------------------------------------------
bow_vec = CountVectorizer(
    tokenizer=tokenize,           # our own tokenizer function
    preprocessor=None,            # skip sklearn's built-in preprocessing (lowercasing, accent stripping, etc.)
    token_pattern=None,           # must be None when using a custom tokenizer, otherwise
    lowercase=False,              # let `tokenize` decide on casing rather than sklearn
    ngram_range=NGRAM_RANGE_BOW,  # Use the parameters defined above
    min_df=MIN_DF,
    max_features=MAX_FEATURES,
)

texts = chunk_df['text'].astype(str).tolist()
# .astype(str) guards against any non-string/NaN values in the column,
# since CountVectorizer expects a list of strings.

# ------------------------------------------------------------
# Step 1: fit — build the vocabulary
# ------------------------------------------------------------
# `fit` runs the tokenizer over every document, counts term frequencies,
# and applies MIN_DF / MAX_FEATURES to decide the final vocabulary.
# At this stage nothing is vectorized, we only learn *which* terms exist.
bow_vec.fit(tqdm(texts, desc="Building vocabulary"))

# ------------------------------------------------------------
# Step 2: transform — vectorize each document against that vocabulary
# ------------------------------------------------------------
# `transform` tokenizes each document again and counts occurrences of
# each vocabulary term, producing one row per document (chunk) and one
# column per vocabulary term. The result, X_bow, is a sparse matrix:
# shape (n_chunks, vocabulary_size).
X_bow = bow_vec.transform(tqdm(texts, desc="Vectorizing (BoW)"))

# ------------------------------------------------------------
# Note: fit + transform can be combined into a single call, fit_transform,
# which is slightly more efficient (one pass instead of two) but hides
# the two conceptual steps above. We split them here for clarity.
# ------------------------------------------------------------
# X_bow = bow_vec.fit_transform(tqdm(texts, desc="Vectorizing (BoW)", total=len(texts)))

bow_terms = np.array(bow_vec.get_feature_names_out())
# The vocabulary itself, as an array of strings
# bow_terms[i] is the term represented by column i of X_bow.

print('Number of chunks:', X_bow.shape[0])
print('Number of features (vocabulary):', X_bow.shape[1])

# Fill the gap

### _What X_bow is_

In [ ]:
# =============================================== YOUR CODE HERE ===============================================
# What is X_bow? Print X_bow type.


In [ ]:
# =============================================== YOUR CODE HERE ===============================================
# What is X_bow's shape? Print X_bow shape.


In [ ]:
# =============================================== YOUR CODE HERE ===============================================
# What is the size of the vocabulary? How many terms does the BoW contain?


## Inspecting frequent terms

We can use our matrix X_bow to perform some sanity check.

Questions to ask:
- Do the frequent terms make sense?
- Are there OCR artifacts or noise tokens?
- Are stopwords still dominating?
- Are philosophically meaningful concepts visible?

In [ ]:
term_counts = np.asarray(X_bow.sum(axis=0)).ravel()

term_df = pd.DataFrame({
    'term': bow_terms,
    'count': term_counts,
})

term_df = term_df.sort_values('count', ascending=False)

display(term_df.head(30))

# n-grams

n-grams extend Bag-of-Words by including short sequences of words.

Examples:
- unigram: `reason`
- bigram: `moral law`
- trigram: `freedom of thought`

Why do this?
Because many concepts are expressed as multi-word expressions rather than isolated tokens.

In [ ]:
# ------------------------------------------------------------
# Vectorization settings
# ------------------------------------------------------------
MIN_DF = 3
MAX_FEATURES = 50000
NGRAM_RANGE_TRIGRAM = (1, 3)
# Unigrams + bigrams + trigrams. A wider context window than the BoW
# model above — captures longer fixed phrases ("categorical imperative
# of reason") at the cost of a larger, sparser vocabulary.

In [ ]:
# ------------------------------------------------------------
# Model 2: extended n-grams (unigrams + bigrams + trigrams)
# ------------------------------------------------------------
# Same settings as bow_vec above, except a wider NGRAM_RANGE_TRIGRAM.
# Comparing X_ngram to X_bow shows how much the vocabulary grows,
# and how much sparser each row becomes, as the n-gram window widens.
ngram_vec = CountVectorizer(
    tokenizer=tokenize,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    ngram_range=NGRAM_RANGE_TRIGRAM,
    min_df=MIN_DF,
    max_features=MAX_FEATURES,
)

# Since fit_transform is slightly more efficient, we combine the two functions here
X_ngram = ngram_vec.fit_transform(tqdm(texts, desc="Vectorizing (n-grams)", total=len(texts)))
ngram_terms = np.array(ngram_vec.get_feature_names_out())

print('\nn-gram matrix shape:', X_ngram.shape) # (n_chunks, vocabulary)
print('Vocabulary size:', len(ngram_terms))

In [ ]:
print(f"Non-zero density: BoW={X_bow.nnz / (X_bow.shape[0]*X_bow.shape[1]):.5%}  "
      f"n-gram={X_ngram.nnz / (X_ngram.shape[0]*X_ngram.shape[1]):.5%}")

In [ ]:
# Check bigrams
bigrams = [f for f in ngram_terms if ' ' in f]
print(f"\nBigrams included: {len(bigrams)}.\n")

In [ ]:
print("\nSample bigrams:\n", bigrams[100:120], '\n')

In [ ]:
ngram_counts = np.asarray(X_ngram.sum(axis=0)).ravel()
ngram_df = pd.DataFrame({
    'term': ngram_terms,
    'count': ngram_counts,
})
ngram_df = ngram_df.sort_values('count', ascending=False)

print("\nTop n-grams:")
print("-" * 60)
display(ngram_df.head(20))

In [ ]:
print("\nTop bigrams:")
print("-" * 60)
display(ngram_df[ngram_df['term'].str.contains(' ')].head(20))

In [ ]:
print("\nTop trigrams:")
print("-" * 60)
display(ngram_df[ngram_df['term'].str.count(' ') == 2].head(20))

### Comparing vocabularies: BoW vs. extended n-grams

`max_features=50000` caps *both* vectorizers at the same vocabulary size, but this does not mean the two vocabularies are the same. The cap is applied across **all** n-gram orders together, so unigrams, bigrams, and (for `ngram_vec`) trigrams are all competing for the same 50,000 slots based on how often they occur in the corpus.

The cell below checks three things:

1. **Did both models actually hit the cap?** If the raw vocabulary (before truncation) is smaller than `max_features`, a model's vocabulary won't reach 50,000 at all.
2. **Composition by n-gram order** — how many unigrams, bigrams, and trigrams ended up in each vocabulary. This tells you how much "room" trigrams actually claimed in `ngram_vec`, and how rare they are relative to shorter n-grams.
3. **Overlap between the two vocabularies** — since both are capped at the same size, every trigram that earns a slot in `ngram_vec` must displace some other term. Comparing the sets shows exactly which (and how many) bigrams got crowded out when trigrams were added to the competition.

Keep this in mind as you interpret the non-zero density numbers from before: a wider n-gram window does notjust make the matrix sparser — it can also quietly reshape *which* terms are being tracked in the first place.

In [ ]:
def ngram_order(term: str) -> int:
    # sklearn joins n-gram tokens with a single space, so counting
    # spaces tells us the n-gram order (0 spaces = unigram, 1 = bigram, ...)
    return term.count(' ') + 1

def ngram_composition(terms: np.ndarray) -> pd.Series:
    orders = pd.Series([ngram_order(t) for t in terms])
    return orders.value_counts().sort_index()

print("Vocabulary sizes: bow =", f"{len(bow_terms):,}", " ngram =", f"{len(ngram_terms):,}")
print("Both hit the max_features cap:",
      len(bow_terms) == MAX_FEATURES, "/", len(ngram_terms) == MAX_FEATURES)

print("\nbow_vec composition (1,2):")
print(ngram_composition(bow_terms).rename_axis("n-gram order").rename("count"))

print("\nngram_vec composition (1,3):")
print(ngram_composition(ngram_terms).rename_axis("n-gram order").rename("count"))

# How much do the two vocabularies actually overlap?
bow_set, ngram_set = set(bow_terms), set(ngram_terms)
overlap = bow_set & ngram_set
print(f"\nShared terms: {len(overlap):,} / {len(bow_set):,} bow terms "
      f"({len(overlap) / len(bow_set):.1%})")
print(f"Terms only in bow_vec (crowded out by trigrams): {len(bow_set - ngram_set):,}")
print(f"Terms only in ngram_vec (trigrams that made the cut): {len(ngram_set - bow_set):,}")

# Critical thinking: inspecting multi-word expressions

- Which philosophical concepts appear as bigrams and trigrams?
- Which n-grams are historically specific?
- Which are too generic to be useful?

 =============================================== YOUR THOUGHTS HERE ===============================================



---

## Advanced TF–IDF for chunk-level classification features

In Notebook 04a, we used TF–IDF as an **exploratory representation**: a way to inspect distinctive vocabulary, compare texts, and think about lexical similarity. In this notebook, we move to a more advanced use of TF–IDF as a **feature-engineering method** for supervised learning.

A key difference here is that we are no longer representing only whole documents. Instead, we apply TF–IDF to **text chunks**, which gives us many more observations and a more fine-grained unit of analysis. As a result, the feature matrix becomes much larger — for example, around **1722 × 50000** in this workflow. This means:

- each **row** represents one chunk rather than one full book
- each **column** represents a vocabulary feature
- the matrix is much more detailed, but also much sparser

This chunk-based design is useful because long philosophical works often contain multiple themes, arguments, and conceptual shifts within the same text. By breaking them into smaller units, we allow the model to capture more localized patterns instead of averaging everything at the book level.

At the same time, this also makes feature design more important. A classifier cannot work directly with raw text: it needs each chunk to be transformed into a numerical representation. TF–IDF is one of the most useful and interpretable ways to do this because it highlights terms that are informative for a given chunk while downweighting terms that are common across the corpus.

In this section, TF–IDF is therefore treated not as an exploratory output, but as a **model input**. We will pay closer attention to parameters such as:

- `min_df`: how rare a term can be before we discard it
- `max_features`: how large we allow the vocabulary to become
- `ngram_range`: whether we use only single words or also multi-word expressions
- tokenization and preprocessing choices that shape the final feature space

These choices matter because they affect what the classifier is able to learn. A chunk-level TF–IDF representation may capture historically distinctive vocabulary, stylistic habits, recurring phrases, or unintended artifacts. For that reason, feature design is not just technical preprocessing: it is part of the methodological argument of the notebook.

In short, the goal here is not only to build a large TF–IDF matrix, but to understand how chunking and representation choices influence downstream classification results.

In [ ]:
# ------------------------------------------------------------
# Vectorization settings
# ------------------------------------------------------------

# Limit the vocabulary to the 30,000 most informative features so the matrix
# stays large enough to be expressive, but not unnecessarily unwieldy.
MAX_FEATURES_TFIDF = 30000

# Include both unigrams, bigrams and trigrams so the model can capture single words
# and short multi-word expressions such as 'natural law' or 'free will'.
NGRAM_RANGE_TFIDF = (1, 3) # Unigram, bigram, trigram - (1, 2) for unigram and bigram only

# Ignore terms that appear in fewer than 5 chunks. This removes many rare,
# noisy, or idiosyncratic terms that are unlikely to generalize well.
MIN_DF_TFIDF = 5

### Updated TF-IDF configuration for historical / philosophical chunks from raw counts to weighted importance

The `bow_vec` and `ngram_vec` matrices above store **raw term counts** — how many times each term appears in each chunk. That treats every occurrence equally, so extremely common words get the same kind of weight as rarer, more informative terms.

Note that stopword removal is already handled upstream, inside the shared `tokenize` function — the same one used by `bow_vec`, `ngram_vec`, and `tfidf_vec` — so common function words (*"the"*, *"and"*, etc.) never even reach the vectorizer. What `TfidfVectorizer` adds on top of that is a *graded* re-weighting: among the terms that do survive tokenization, ones that are frequent within a chunk but rare *across the corpus* get boosted, while ones that appear in almost every chunk get down-weighted — capturing a distinction that raw counts (and stopword lists) can't, since stopword removal is all-or-nothing but TF–IDF weighting is continuous.

This configuration reuses ideas from the BoW/n-gram comparison above, with a few adjustments:

- **`NGRAM_RANGE = (1, 3)`** — same unigram+bigram+trigram range as `ngram_vec`, so multi-word expressions like *"natural law"* or *"categorical imperative"* are still captured as single features.
- **`MAX_FEATURES = 30000`** — a tighter cap than before (50,000), and **`MIN_DF = 5`** is stricter than the `MIN_DF = 3` used earlier. Combined with a smaller `max_features`, this vocabulary will end up both smaller *and* more selective than `bow_vec`'s or `ngram_vec`'s — worth checking with the same composition/overlap code from before if you want to see exactly how much it shrinks and which terms survive.
- **`strip_accents="unicode"`** — normalizes accented characters, useful for reducing spurious mismatches across historical spelling variants.
- **`sublinear_tf=True`** — replaces raw term frequency with `1 + log(tf)`, so a term appearing 20 times in a chunk doesn't count 20× as much as a term appearing once; it dampens the effect of repetition bursts.
- **`norm='l2'`** — scales each chunk's vector to unit length, the standard normalization for cosine-similarity-based comparisons (which we'll use later).

The goal is the same as before — comparing how different vectorization choices reshape the resulting feature space — but now weighting is part of the comparison too, not just vocabulary size and n-gram range.

In [ ]:
tfidf_vec = TfidfVectorizer(
    tokenizer=tokenize,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    # Normalize accented characters to reduce minor spelling variation and
    # improve matching across historical texts.
    strip_accents="unicode",
    ngram_range=NGRAM_RANGE_TFIDF,
    min_df=MIN_DF_TFIDF,
    max_features=MAX_FEATURES_TFIDF,
    # Replace raw term frequency with a log-scaled version to reduces the
    # impact of repeated bursts of the same term within a single chunk.
    sublinear_tf=True,
    # Normalize each vector to unit length, which is standard for cosine-based
    # comparison and works well for many classification workflows.
    norm='l2',
)

T = tfidf_vec.fit_transform(tqdm(chunk_df['text'].astype(str).tolist(), total=len(chunk_df['text'])))

terms = np.array(tfidf_vec.get_feature_names_out())

print('\nTF–IDF matrix shape:', T.shape)
print('Vocabulary size:', len(terms))

## Inspecting top TF–IDF terms for a document

Unlike raw counts, TF–IDF highlights terms that are especially characteristic of a particular text.

Questions:
- Which terms define this document?
- Are the results interpretable?
- Do bigrams improve interpretability?

In [ ]:
def top_terms_for_doc(i: int, k: int = 15):
    row = T.getrow(i)

    if row.nnz == 0:
        return []

    order = np.argsort(-row.data)[:k]
    idx = row.indices[order]

    return list(zip(terms[idx], row.data[order]))

# Example document
i0 = np.random.randint(0, len(df))

print(chunk_df.iloc[i0][['pg_id', 'title', 'time_bin']].to_dict())
print()

for term, score in top_terms_for_doc(i0, k=20):
    print(f'{term:<30} {score:.4f}')

# Similarity between documents

Documents represented in TF–IDF space can be compared with cosine similarity.

Questions:
- Which books are lexically similar?
- Do similar texts come from the same period?
- Are there cross-period similarities?

In [ ]:
def doc_neighbors(df: pd.DataFrame, i:int, k:int = 5, snippet_len=200):
    """
    Find the k nearest neighbours of document i using cosine similarity.
    
    Parameters:
        df : pd.DataFrame – must have a 'text' column with the raw text.
        i : int – index of the query document.
        k : int – number of neighbours (excluding the query itself).
        snippet_len : int – length of text snippet to display.
    Returns:
        pd.DataFrame with columns: row, cosine, pg_id, title, time_bin, snippet
    """
    sims = cosine_similarity(T[i], T).ravel()
    
    # Sort descending, take k+1 (including self)
    nn = np.argsort(-sims)[:k+1]
    
    rows = []
    for j in nn:
        # Get text snippet
        text = df.iloc[j]['text']
        snippet = text[:snippet_len] + ('…' if len(text) > snippet_len else '')
        rows.append({
            'row': int(j),
            'cosine': float(sims[j]),
            'pg_id': int(df.iloc[j]['pg_id']),
            'title': df.iloc[j]['title'],
            'time_bin': df.iloc[j]['time_bin'],
            'snippet': snippet,
        })
    
    return pd.DataFrame(rows)

In [ ]:
# Query document index (e.g. i=0)
neighbors_df = doc_neighbors(chunk_df, i=0, k=5)

# Show the query text
print("\nQUERY DOCUMENT:")
print(chunk_df.iloc[0]['title'] + "\n")

# Show neighbours
print("NEAREST NEIGHBOURS:")
display(neighbors_df[['title', 'cosine', 'time_bin', 'snippet']])

# Fill the gap

In [ ]:
# =============================================== YOUR CODE HERE ===============================================
# Eplore further neighbours with the function `doc_neighbors`

# Time-bin centroids

We can average document vectors within a historical period to obtain a centroid representation of that period.

This allows us to ask:
- Which terms characterize a period?
- Which periods look lexically similar?
- Which concepts persist or disappear over time?

In [ ]:
# ------------------------------------------------------------
# Helper function to sort years chronologically
# ------------------------------------------------------------
def bin_start(label) -> int:
    s = str(label)
    m = re.search(r'-?\d+', s.replace('–', '-'))
    return int(m.group(0)) if m else 10**9

In [ ]:
TOP_N = 12

# ------------------------------------------------------------
# Build top terms per time bin
# ------------------------------------------------------------
bin_terms = []

valid_bins = chunk_df.dropna(subset=['time_bin']).copy()
time_order = sorted(valid_bins['time_bin'].astype(str).unique().tolist(), key=bin_start)

for bin_label, idx in valid_bins.groupby('time_bin').indices.items():
    mean_vec = np.asarray(T[idx].mean(axis=0)).ravel()
    top_idx = np.argsort(-mean_vec)[:TOP_N]

    for j in top_idx:
        bin_terms.append({
            'time_bin': str(bin_label),
            'term': terms[j],
            'mean_tfidf': float(mean_vec[j]),
        })

bin_terms = pd.DataFrame(bin_terms)

display(bin_terms.head(30))

# ------------------------------------------------------------
# Plot as small multiples: one panel per period
# ------------------------------------------------------------
n_bins = len(time_order)
ncols = 2
nrows = math.ceil(n_bins / ncols)

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(16, max(4 * nrows, 6)),
    squeeze=False
)

for ax, bin_label in zip(axes.flat, time_order):
    sub = (
        bin_terms[bin_terms['time_bin'] == bin_label]
        .sort_values('mean_tfidf', ascending=True)
    )

    sns.barplot(
        data=sub,
        x='mean_tfidf',
        y='term',
        ax=ax,
        color='teal'
    )
    ax.set_title(f"Top {TOP_N} TF-IDF terms: {bin_label}")
    ax.set_xlabel("Mean TF-IDF")
    ax.set_ylabel("")

for ax in axes.flat[n_bins:]:
    ax.set_visible(False)

plt.suptitle("Top weighted TF-IDF terms by time period", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
TOP_N = 10

bin_terms = []

valid_bins = chunk_df.dropna(subset=['time_bin']).copy()
time_order = sorted(valid_bins['time_bin'].astype(str).unique().tolist(), key=bin_start)

for bin_label, idx in valid_bins.groupby('time_bin').indices.items():
    mean_vec = np.asarray(T[idx].mean(axis=0)).ravel()
    top_idx = np.argsort(-mean_vec)[:TOP_N]

    for j in top_idx:
        bin_terms.append({
            'time_bin': str(bin_label),
            'term': terms[j],
            'mean_tfidf': float(mean_vec[j]),
        })

bin_terms = pd.DataFrame(bin_terms)

# union of all top terms across bins
top_term_union = bin_terms['term'].drop_duplicates().tolist()

heatmap_rows = []
for bin_label, idx in valid_bins.groupby('time_bin').indices.items():
    mean_vec = np.asarray(T[idx].mean(axis=0)).ravel()
    term_to_score = {terms[j]: float(mean_vec[j]) for j in range(len(terms))}

    for term in top_term_union:
        heatmap_rows.append({
            'time_bin': str(bin_label),
            'term': term,
            'mean_tfidf': term_to_score.get(term, 0.0),
        })

heatmap_df = pd.DataFrame(heatmap_rows)
heatmap_pivot = heatmap_df.pivot(index='term', columns='time_bin', values='mean_tfidf')
heatmap_pivot = heatmap_pivot.reindex(columns=time_order)
heatmap_pivot = heatmap_pivot.loc[heatmap_pivot.max(axis=1).sort_values(ascending=False).index]

plt.figure(figsize=(12, max(8, 0.35 * len(heatmap_pivot))))
sns.heatmap(heatmap_pivot, cmap='YlGnBu', linewidths=0.4)
plt.title("Top TF-IDF terms across time periods")
plt.xlabel("Time period")
plt.ylabel("Term")
plt.tight_layout()
plt.show()

## Comparing time periods with TF–IDF centroid similarity

A useful way to summarize chunk-level TF–IDF patterns is to compute an **average TF–IDF vector** for each `time_bin` and then compare those averages with one another. These average vectors are often called **centroids**.

The heatmap below shows the **cosine similarity** between time-bin centroids. High values indicate that two periods have relatively similar lexical profiles under TF–IDF weighting, while lower values suggest greater lexical difference.

This visualization is helpful because it shifts attention from individual chunks to the **overall vocabulary profile of a period**. It can therefore support broader questions about continuity and change across time, while remaining grounded in the TF–IDF representation used later for modeling.

In [ ]:
valid = chunk_df.dropna(subset=['time_bin']).copy()
time_order = sorted(valid['time_bin'].astype(str).unique().tolist(), key=bin_start)

centroids = []
labels = []
counts = []

for bin_label, idx in valid.groupby('time_bin').indices.items():
    idx = np.array(list(idx), dtype=int)
    centroid = np.asarray(T[idx].mean(axis=0))
    centroids.append(centroid)
    labels.append(str(bin_label))
    counts.append(len(idx))

C = np.vstack(centroids)
S = cosine_similarity(C)

sim_df = pd.DataFrame(S, index=labels, columns=labels)
sim_df = sim_df.reindex(index=time_order, columns=time_order)

plt.figure(figsize=(10, 8))
sns.heatmap(sim_df, cmap='crest', vmin=0, vmax=1, annot=True, fmt='.2f')
plt.title('TF-IDF centroid similarity across time bins')
plt.xlabel('Time bin')
plt.ylabel('Time bin')
plt.tight_layout()
plt.show()

pd.DataFrame({'time_bin': labels, 'n_chunks': counts}).sort_values('time_bin', key=lambda s: s.map(bin_start))

## Inspecting sparsity in the chunk-level TF–IDF matrix

Because TF–IDF is built over a very large vocabulary, most chunks contain only a small fraction of all possible features. This means that the TF–IDF matrix is **sparse**: most cells are zero.

The plots below help us inspect that sparsity directly. The histogram shows how many TF–IDF features are active in a typical chunk, while the boxplot compares this distribution across time bins.

These diagnostics are useful because they make the feature space more concrete. Instead of treating TF–IDF as an abstract matrix, we can see how densely or sparsely chunks are represented, and whether some periods contain systematically richer or narrower lexical profiles than others.

In [ ]:
# Calculate the number of non-zero features
nnz_per_chunk = np.asarray((T > 0).sum(axis=1)).ravel()

plot_df = chunk_df.copy().reset_index(drop=True)
plot_df['nnz_features'] = nnz_per_chunk

time_order = sorted(
    plot_df['time_bin'].dropna().astype(str).unique().tolist(),
    key=bin_start
)

plot_df['time_bin'] = pd.Categorical(
    plot_df['time_bin'].astype(str),
    categories=time_order,
    ordered=True
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(
    plot_df['nnz_features'],
    bins=30,
    kde=True,
    ax=axes[0],
    color='teal'
)
axes[0].set_title('Distribution of nonzero TF-IDF features per chunk')
axes[0].set_xlabel('Number of nonzero features')
axes[0].set_ylabel('Number of chunks')

sns.boxplot(
    data=plot_df.dropna(subset=['time_bin']).sort_values('time_bin'),
    x='time_bin',
    y='nnz_features',
    ax=axes[1],
    color='gold'
)
axes[1].set_title('Nonzero TF-IDF features per chunk by time bin')
axes[1].set_xlabel('Time bin')
axes[1].set_ylabel('Number of nonzero features')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print('TF-IDF matrix shape:', T.shape)
print('Average nonzero features per chunk:', round(plot_df['nnz_features'].mean(), 1))
print('Median nonzero features per chunk:', int(plot_df['nnz_features'].median()))
print('Matrix density:', round(T.nnz / (T.shape[0] * T.shape[1]), 6))

## Comparing fixed-width and quantile time bins

A key methodological choice in diachronic text analysis is how we divide time into periods. In the earlier version of this notebook, we used **fixed-width bins** of 500 years. This approach is easy to interpret historically because each bin corresponds to the same chronological span. However, it can also create highly uneven distributions: some bins may contain many texts or tokens, while others contain very little material.

To explore this issue, we now compare the fixed-width approach with a **quantile-based binning scheme**. Quantile bins do not preserve equal chronological width. Instead, they divide the corpus so that each bin contains a more similar amount of material. This can make comparisons more balanced, especially when the corpus is unevenly distributed across centuries.

The goal of this comparison is not to decide that one binning method is universally better. Rather, it is to examine how different binning choices shape the patterns we see. If document counts, token counts, or TF–IDF summaries change substantially under a different scheme, that is a reminder that temporal analysis is partly a modeling decision, not simply a neutral description of history.

In [ ]:
# ------------------------------------------------------------------
# Outputs:
# - df_timebins_from_docs
# - timebin_compare_from_docs
# - saved CSVs + one comparison figure
# ------------------------------------------------------------------

FIXED_WIDTH = 500
N_QUANTILES = 6

def coerce_year_value(value):
    if value is None or pd.isna(value):
        return pd.NA
    s = str(value).replace("−", "-")
    m = re.search(r"-?\d{1,4}", s)
    return int(m.group(0)) if m else pd.NA


# ------------------------------------------------------------------
# Build dataframe from spaCy docs rather than the main df
# ------------------------------------------------------------------
rows = []
for i, doc in enumerate(docs):
    meta = doc.user_data
    pub_year = (
        meta.get("publication_year")
        if "publication_year" in meta
        else meta.get("analysis_year", meta.get("year"))
    )
    rows.append({
        "doc_i": i,
        "pg_id": meta.get("pg_id"),
        "filename": meta.get("filename"),
        "title": meta.get("title"),
        "publication_year": coerce_year_value(pub_year),
        "n_tokens": len(doc),
        "text": doc.text,
    })


df_timebins_from_docs = pd.DataFrame(rows)
df_timebins_from_docs = df_timebins_from_docs.dropna(subset=["publication_year"]).copy()
df_timebins_from_docs["publication_year"] = pd.to_numeric(
    df_timebins_from_docs["publication_year"], errors="coerce"
).astype("Int64")

if df_timebins_from_docs.empty:
    raise ValueError("No valid publication_year values found in doc.user_data.")


def time_bins_fixed(year: pd.Series, width: int = 500) -> pd.Categorical:
    y = year.dropna().astype(int)
    lo, hi = int(y.min()), int(y.max())
    start = (lo // width) * width
    end = ((hi // width) + 1) * width
    bins = list(range(start, end + 1, width))
    labels = [f"{bins[i]}–{bins[i + 1] - 1}" for i in range(len(bins) - 1)]
    return pd.cut(year.astype(float), bins=bins, labels=labels, include_lowest=True)


def time_bins_quantile(year: pd.Series, q: int = 6) -> pd.Categorical:
    binned = pd.qcut(year.astype(float), q=q, duplicates="drop")
    if hasattr(binned, "cat"):
        labels = [
            f"{int(round(iv.left))}–{int(round(iv.right))}"
            for iv in binned.cat.categories
        ]
        binned = binned.cat.rename_categories(labels)
    return binned


def summarize_bins(frame: pd.DataFrame, bin_col: str, scheme_name: str) -> pd.DataFrame:
    out = (
        frame.dropna(subset=[bin_col])
        .assign(time_bin=lambda d: d[bin_col].astype(str))
        .groupby("time_bin", as_index=False)
        .agg(
            n_docs=("doc_i", "size"),
            total_tokens=("n_tokens", "sum"),
            median_year=("publication_year", "median"),
            min_year=("publication_year", "min"),
            max_year=("publication_year", "max"),
        )
    )
    out["scheme"] = scheme_name
    return out


df_timebins_from_docs["time_bin_fixed500"] = time_bins_fixed(
    df_timebins_from_docs["publication_year"], width=FIXED_WIDTH
)
df_timebins_from_docs["time_bin_quantile"] = time_bins_quantile(
    df_timebins_from_docs["publication_year"], q=N_QUANTILES
)

fixed_summary = summarize_bins(df_timebins_from_docs, "time_bin_fixed500", "Fixed 500-year bins")
quant_summary = summarize_bins(df_timebins_from_docs, "time_bin_quantile", "Quantile bins")

timebin_compare_from_docs = pd.concat([fixed_summary, quant_summary], ignore_index=True)

df_timebins_from_docs.to_csv(TABLES_DIR / "nb07-df_timebins_from_spacy_docs.csv", index=False)
timebin_compare_from_docs.to_csv(TABLES_DIR / "nb07-timebin_compare_from_spacy_docs.csv", index=False)

fig, axes = plt.subplots(2, 2, figsize=(16, 10), constrained_layout=True)

for ax, sub, title, value_col, color in [
    (axes[0, 0], fixed_summary, "Fixed 500-year bins: documents per bin", "n_docs", "#4C72B0"),
    (axes[0, 1], quant_summary, "Quantile bins: documents per bin", "n_docs", "#55A868"),
    (axes[1, 0], fixed_summary, "Fixed 500-year bins: tokens per bin", "total_tokens", "#C44E52"),
    (axes[1, 1], quant_summary, "Quantile bins: tokens per bin", "total_tokens", "#8172B2"),
]:
    plot_df = sub.sort_values("median_year").copy()
    sns.barplot(data=plot_df, x="time_bin", y=value_col, ax=ax, color=color)
    ax.set_title(title)
    ax.set_xlabel("Time bin")
    ax.set_ylabel("Documents" if value_col == "n_docs" else "Total tokens")
    ax.tick_params(axis="x", rotation=45)

fig.suptitle("Comparing fixed-width and quantile time bins from spaCy docs", fontsize=14, y=1.02)
fig_path = FIGURES_DIR / "timebin_compare_from_spacy_docs.png"
plt.savefig(fig_path, dpi=250, bbox_inches="tight")
plt.show()

print("Saved table:", TABLES_DIR / "nb07-df_timebins_from_spacy_docs.csv")
print("Saved table:", TABLES_DIR / "nb07-timebin_compare_from_spacy_docs.csv")
print("Saved figure:", fig_path)
display(timebin_compare_from_docs.sort_values(["scheme", "median_year"]))


In [ ]:
# ------------------------------------------------------------------
# Outputs:
# - df_timebins_from_docs
# - timebin_compare_from_docs
# - saved CSVs + one comparison figure
# ------------------------------------------------------------------

FIXED_WIDTH = 500
N_QUANTILES = 6

# ------------------------------------------------------------------
# Build dataframe
# ------------------------------------------------------------------

df_timebins_from_docs = chunk_df.dropna(subset=["publication_year"]).copy()
df_timebins_from_docs["publication_year"] = pd.to_numeric(
    df_timebins_from_docs["publication_year"], errors="coerce"
).astype("Int64")

if df_timebins_from_docs.empty:
    raise ValueError("No valid publication_year values found in doc.user_data.")


def time_bins_fixed(year: pd.Series, width: int = 500) -> pd.Categorical:
    y = year.dropna().astype(int)
    lo, hi = int(y.min()), int(y.max())
    start = (lo // width) * width
    end = ((hi // width) + 1) * width
    bins = list(range(start, end + 1, width))
    labels = [f"{bins[i]}–{bins[i + 1] - 1}" for i in range(len(bins) - 1)]
    return pd.cut(year.astype(float), bins=bins, labels=labels, include_lowest=True)


def time_bins_quantile(year: pd.Series, q: int = 6) -> pd.Categorical:
    binned = pd.qcut(year.astype(float), q=q, duplicates="drop")
    if hasattr(binned, "cat"):
        labels = [
            f"{int(round(iv.left))}–{int(round(iv.right))}"
            for iv in binned.cat.categories
        ]
        binned = binned.cat.rename_categories(labels)
    return binned

def summarize_bins(frame: pd.DataFrame, bin_col: str, scheme_name: str) -> pd.DataFrame:
    out = (
        frame.dropna(subset=[bin_col])
        .assign(time_bin=lambda d: d[bin_col].astype(str))
        .groupby("time_bin", as_index=False)
        .agg(
            n_docs=("chunk_index", "size"),
            total_tokens=("n_tokens", "sum"),
            median_year=("publication_year", "median"),
            min_year=("publication_year", "min"),
            max_year=("publication_year", "max"),
        )
    )
    out["scheme"] = scheme_name
    return out


df_timebins_from_docs["time_bin_fixed500"] = time_bins_fixed(
    df_timebins_from_docs["publication_year"], width=FIXED_WIDTH
)
df_timebins_from_docs["time_bin_quantile"] = time_bins_quantile(
    df_timebins_from_docs["publication_year"], q=N_QUANTILES
)

fixed_summary = summarize_bins(df_timebins_from_docs, "time_bin_fixed500", "Fixed 500-year bins")
quant_summary = summarize_bins(df_timebins_from_docs, "time_bin_quantile", "Quantile bins")

timebin_compare_from_docs = pd.concat([fixed_summary, quant_summary], ignore_index=True)

df_timebins_from_docs.to_csv(TABLES_DIR / "nb07-df_timebins_from_spacy_docs.csv", index=False)
timebin_compare_from_docs.to_csv(TABLES_DIR / "nb07-timebin_compare_from_spacy_docs.csv", index=False)

fig, axes = plt.subplots(2, 2, figsize=(16, 10), constrained_layout=True)

for ax, sub, title, value_col, color in [
    (axes[0, 0], fixed_summary, "Fixed 500-year bins: documents per bin", "n_docs", "#4C72B0"),
    (axes[0, 1], quant_summary, "Quantile bins: documents per bin", "n_docs", "#55A868"),
    (axes[1, 0], fixed_summary, "Fixed 500-year bins: tokens per bin", "total_tokens", "#C44E52"),
    (axes[1, 1], quant_summary, "Quantile bins: tokens per bin", "total_tokens", "#8172B2"),
]:
    plot_df = sub.sort_values("median_year").copy()
    sns.barplot(data=plot_df, x="time_bin", y=value_col, ax=ax, color=color)
    ax.set_title(title)
    ax.set_xlabel("Time bin")
    ax.set_ylabel("Documents" if value_col == "n_docs" else "Total tokens")
    ax.tick_params(axis="x", rotation=45)

fig.suptitle("Comparing fixed-width and quantile time bins from spaCy docs", fontsize=14, y=1.02)
fig_path = FIGURES_DIR / "timebin_compare_from_spacy_docs.png"
plt.savefig(fig_path, dpi=250, bbox_inches="tight")
plt.show()

print("Saved table:", TABLES_DIR / "nb07-df_timebins_from_spacy_docs.csv")
print("Saved table:", TABLES_DIR / "nb07-timebin_compare_from_spacy_docs.csv")
print("Saved figure:", fig_path)
display(timebin_compare_from_docs.sort_values(["scheme", "median_year"]))


In [ ]:
# ------------------------------------------------------------------
# Outputs:
# - df_tfidf_bins_from_docs
# - T_2
# - terms
# ------------------------------------------------------------------

FIXED_WIDTH = 500
N_QUANTILES = 6
MAX_FEATURES = 30000
MIN_DF = 3
NGRAM_RANGE = (1, 2)


def time_bins_fixed(year: pd.Series, width: int = 500) -> pd.Categorical:
    y = year.dropna().astype(int)
    if y.empty:
        raise ValueError("No valid years found to create fixed time bins.")
    lo, hi = int(y.min()), int(y.max())
    start = (lo // width) * width
    end = ((hi // width) + 1) * width
    bins = list(range(start, end + 1, width))
    labels = [f"{bins[i]}–{bins[i + 1] - 1}" for i in range(len(bins) - 1)]
    return pd.cut(year.astype(float), bins=bins, labels=labels, include_lowest=True)


def time_bins_quantile(year: pd.Series, q: int = 6) -> pd.Categorical:
    valid = year.dropna().astype(float)
    if valid.empty:
        raise ValueError("No valid years found to create quantile time bins.")
    binned = pd.qcut(year.astype(float), q=q, duplicates="drop")
    if hasattr(binned, "cat"):
        labels = [
            f"{int(round(iv.left))}–{int(round(iv.right))}"
            for iv in binned.cat.categories
        ]
        binned = binned.cat.rename_categories(labels)
    return binned

def sort_bin_labels(values) -> list[str]:
    vals = [str(v) for v in values if pd.notna(v)]
    return sorted(set(vals), key=bin_start)


# ------------------------------------------------------------------
# Build dataframe
# ------------------------------------------------------------------

df_tfidf_bins_from_docs = (
    chunk_df
    .dropna(subset=["publication_year"])
    .copy()
    .reset_index(drop=True)
)

df_tfidf_bins_from_docs["publication_year"] = pd.to_numeric(
    df_tfidf_bins_from_docs["publication_year"], errors="coerce"
).astype("Int64")

if df_tfidf_bins_from_docs.empty:
    raise ValueError("No valid publication_year values found in docs.user_data.")


df_tfidf_bins_from_docs["time_bin_fixed500"] = time_bins_fixed(
    df_tfidf_bins_from_docs["publication_year"], width=FIXED_WIDTH
)
df_tfidf_bins_from_docs["time_bin_quantile"] = time_bins_quantile(
    df_tfidf_bins_from_docs["publication_year"], q=N_QUANTILES
)

print("Chunks with usable publication year:", len(df_tfidf_bins_from_docs))
print(
    "Fixed bins:",
    sort_bin_labels(df_tfidf_bins_from_docs["time_bin_fixed500"].dropna().unique())
)
print(
    "Quantile bins:",
    sort_bin_labels(df_tfidf_bins_from_docs["time_bin_quantile"].dropna().unique())
)

STOPWORDS_LIST = sorted(set(w.lower() for w in STOPWORDS))

# ------------------------------------------------------------------
# Compute one shared TF-IDF space for all chunks
# ------------------------------------------------------------------
vectorizer = TfidfVectorizer(
    lowercase=True,
    # Convert all text to lowercase before vectorization so that 'Reason'
    # and 'reason' are treated as the same feature.

    stop_words=STOPWORDS_LIST,
    # Use the custom stop word list instead of the generic built-in English list

    strip_accents="unicode",
    # Normalize accented characters to reduce minor spelling variation and
    # improve matching across historical texts.

    min_df=5,
    # Ignore terms that appear in fewer than 5 chunks. This removes many rare,
    # noisy, or idiosyncratic terms that are unlikely to generalize well.

    max_df=0.60,
    # Ignore terms that appear in more than 60% of chunks. These are often too
    # common to be analytically useful, even if they are not in the stopword list.

    max_features=30000,
    # Limit the vocabulary to the 30,000 most informative features so the matrix
    # stays large enough to be expressive, but not unnecessarily unwieldy.

    ngram_range=(1, 2),
    # Include both unigrams and bigrams so the model can capture single words
    # and short multi-word expressions such as 'natural law' or 'free will'.

    sublinear_tf=True,
    # Replace raw term frequency with a log-scaled version. This reduces the
    # impact of repeated bursts of the same term within a single chunk.

    norm="l2",
    # Normalize each vector to unit length, which is standard for cosine-based
    # comparison and works well for many classification workflows.
)

T_2 = vectorizer.fit_transform(df_tfidf_bins_from_docs["text"].astype(str))
terms = np.asarray(vectorizer.get_feature_names_out())

print("TF-IDF matrix shape:", T_2.shape)

display(
    df_tfidf_bins_from_docs[
        ["chunk_index", "pg_id", "publication_year", "time_bin_fixed500", "time_bin_quantile", "n_tokens"]
    ].head()
)

In [ ]:
def top_terms_by_bin(frame: pd.DataFrame, bin_col: str, top_n: int = 12) -> pd.DataFrame:
    rows = []
    bin_values = frame[bin_col].astype(str).to_numpy()

    for bin_label in sort_bin_labels(frame[bin_col].dropna().unique()):
        idx = np.where(bin_values == str(bin_label))[0]
        if len(idx) == 0:
            continue

        mean_vec = np.asarray(T_2[idx].mean(axis=0)).ravel()
        top_idx = np.argsort(-mean_vec)[:top_n]

        for rank, j in enumerate(top_idx, start=1):
            rows.append({
                "time_bin": str(bin_label),
                "rank": rank,
                "term": terms[j],
                "mean_tfidf": float(mean_vec[j]),
            })

    out = pd.DataFrame(rows)
    return out

def centroid_similarity_by_bin(frame: pd.DataFrame, bin_col: str) -> pd.DataFrame:
    labels = sort_bin_labels(frame[bin_col].dropna().astype(str).unique())
    centroids = []
    keep_labels = []

    bin_values = frame[bin_col].astype(str).to_numpy()

    for label in labels:
        idx = np.where(bin_values == label)[0]
        if len(idx) == 0:
            continue
        centroid = np.asarray(T_2[idx].mean(axis=0)).reshape(1, -1)
        centroids.append(centroid)
        keep_labels.append(label)

    if not centroids:
        return pd.DataFrame()

    C = np.vstack(centroids)
    S = cosine_similarity(C)
    return pd.DataFrame(S, index=keep_labels, columns=keep_labels)


def build_top_term_heatmap(top_terms_df: pd.DataFrame, frame: pd.DataFrame, bin_col: str) -> pd.DataFrame:
    labels = sort_bin_labels(frame[bin_col].dropna().astype(str).unique())
    top_term_union = top_terms_df["term"].drop_duplicates().tolist()

    rows = []
    bin_values = frame[bin_col].astype(str).to_numpy()

    for label in labels:
        idx = np.where(bin_values == label)[0]
        if len(idx) == 0:
            continue

        mean_vec = np.asarray(T_2[idx].mean(axis=0)).ravel()
        score_map = {terms[j]: float(mean_vec[j]) for j in range(len(terms))}

        for term in top_term_union:
            rows.append({
                "time_bin": label,
                "term": term,
                "mean_tfidf": score_map.get(term, 0.0),
            })

    heat = pd.DataFrame(rows).pivot(index="term", columns="time_bin", values="mean_tfidf")
    heat = heat.reindex(columns=labels)
    heat = heat.loc[heat.max(axis=1).sort_values(ascending=False).index]
    return heat

In [ ]:
# Parameter
TOP_TERMS_PER_BIN = 12

fixed_top_terms_df = top_terms_by_bin(df_tfidf_bins_from_docs, "time_bin_fixed500", top_n=TOP_TERMS_PER_BIN)
quant_top_terms_df = top_terms_by_bin(df_tfidf_bins_from_docs, "time_bin_quantile", top_n=TOP_TERMS_PER_BIN)

fixed_centroid_sim = centroid_similarity_by_bin(df_tfidf_bins_from_docs, "time_bin_fixed500")
quant_centroid_sim = centroid_similarity_by_bin(df_tfidf_bins_from_docs, "time_bin_quantile")

fixed_top_heat = build_top_term_heatmap(fixed_top_terms_df, df_tfidf_bins_from_docs, "time_bin_fixed500")
quant_top_heat = build_top_term_heatmap(quant_top_terms_df, df_tfidf_bins_from_docs, "time_bin_quantile")

# ------------------------------------------------------------------
# Save tables
# ------------------------------------------------------------------
df_tfidf_bins_from_docs.to_csv(TABLES_DIR / "nb07-df_tfidf_timebins_from_spacy_docs.csv", index=False)
fixed_top_terms_df.to_csv(TABLES_DIR / "nb07-tfidf_top_terms_fixed500_from_spacy_docs.csv", index=False)
quant_top_terms_df.to_csv(TABLES_DIR / "nb07-tfidf_top_terms_quantile_from_spacy_docs.csv", index=False)
fixed_centroid_sim.to_csv(TABLES_DIR / "nb07-tfidf_centroid_similarity_fixed500_from_spacy_docs.csv")
quant_centroid_sim.to_csv(TABLES_DIR / "nb07-tfidf_centroid_similarity_quantile_from_spacy_docs.csv")
fixed_top_heat.to_csv(TABLES_DIR / "nb07-tfidf_top_terms_heatmap_fixed500_from_spacy_docs.csv")
quant_top_heat.to_csv(TABLES_DIR / "nb07-tfidf_top_terms_heatmap_quantile_from_spacy_docs.csv")

# ------------------------------------------------------------------
# Visual comparison: top-term heatmaps + centroid similarities
# ------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(18, 14), constrained_layout=True)

sns.heatmap(fixed_top_heat, cmap="crest", linewidths=0.3, ax=axes[0, 0])
axes[0, 0].set_title(f"Top TF-IDF terms by fixed {FIXED_WIDTH}-year bins")
axes[0, 0].set_xlabel("Time bin")
axes[0, 0].set_ylabel("Term")

sns.heatmap(quant_top_heat, cmap="crest", linewidths=0.3, ax=axes[0, 1])
axes[0, 1].set_title(f"Top TF-IDF terms by quantile bins (q={N_QUANTILES})")
axes[0, 1].set_xlabel("Time bin")
axes[0, 1].set_ylabel("Term")

sns.heatmap(fixed_centroid_sim, cmap="crest", vmin=0, vmax=1, annot=True, fmt=".2f", ax=axes[1, 0])
axes[1, 0].set_title("TF-IDF centroid similarity: fixed bins")
axes[1, 0].set_xlabel("Time bin")
axes[1, 0].set_ylabel("Time bin")

sns.heatmap(quant_centroid_sim, cmap="crest", vmin=0, vmax=1, annot=True, fmt=".2f", ax=axes[1, 1])
axes[1, 1].set_title("TF-IDF centroid similarity: quantile bins")
axes[1, 1].set_xlabel("Time bin")
axes[1, 1].set_ylabel("Time bin")

fig.suptitle("How TF-IDF summaries change under fixed-width vs quantile time bins", fontsize=16, y=1.02)
fig_path = FIGURES_DIR / "nb07-tfidf_fixed_vs_quantile_timebin_comparison.png"
plt.savefig(fig_path, dpi=250, bbox_inches="tight")
plt.show()

print("Saved table:", TABLES_DIR / "nb07-df_tfidf_timebins_from_spacy_docs.csv")
print("Saved table:", TABLES_DIR / "nb07-tfidf_top_terms_fixed500_from_spacy_docs.csv")
print("Saved table:", TABLES_DIR / "nb07-tfidf_top_terms_quantile_from_spacy_docs.csv")
print("Saved table:", TABLES_DIR / "nb07-tfidf_centroid_similarity_fixed500_from_spacy_docs.csv")
print("Saved table:", TABLES_DIR / "nb07-tfidf_centroid_similarity_quantile_from_spacy_docs.csv")
print("Saved figure:", fig_path)

display(fixed_top_terms_df.head(20))
display(quant_top_terms_df.head(20))

# Saving reusable feature artifacts

We save the matrices and vocabularies so later notebooks can reuse them without recomputing vectorization.

This is important for:
- reproducibility
- CPU efficiency
- stable downstream classification workflows

In [ ]:
# Save sparse matrices
sparse.save_npz(CACHE_DIR / 'nb07-bow_matrix.npz', X_bow)
sparse.save_npz(CACHE_DIR / 'nb07-ngram_matrix.npz', X_ngram)
sparse.save_npz(CACHE_DIR / 'nb07-tfidf_matrix.npz', T)
sparse.save_npz(CACHE_DIR / 'nb07-tfidf_matrix_2.npz', T_2)

# Save vocabularies
pd.Series(bow_terms).to_csv(TABLES_DIR / 'nb07-bow_terms.csv', index=False)
pd.Series(ngram_terms).to_csv(TABLES_DIR / 'nb07-ngram_terms.csv', index=False)
pd.Series(terms).to_csv(TABLES_DIR / 'nb07-tfidf_terms.csv', index=False)

# Save top TF–IDF terms by bin
bin_terms.to_csv(TABLES_DIR / 'nb07-tfidf_top_terms_by_bin.csv', index=False)

print('Saved feature artifacts.')

## Critical thinking and reflection

This comparison raises an important methodological question: when we describe patterns “over time,” what exactly are we comparing?

A fixed-width bin treats history as a sequence of equal chronological intervals, even if the surviving corpus is very uneven across those intervals. A quantile bin treats time more flexibly, prioritizing balance in the amount of material, but potentially grouping together periods that are historically quite different.

As you reflect on the outputs from this notebook, consider the following questions:

- Which binning scheme seems more appropriate for your research goal: historical interpretability or analytical balance?
- Do the most distinctive TF–IDF terms remain stable across the two schemes, or do they change noticeably?
- If a pattern appears only under one binning strategy, should we treat it as a robust result or as a binning artifact?
- How might uneven corpus composition — for example, more texts from some periods than others — affect claims about conceptual change?

A useful takeaway is that **time bins do not merely organize the data; they help produce the patterns we interpret**. Good historical NLP practice therefore requires not only reporting results, but also reflecting on how analytical choices shape those results.

## Conclusion

In this notebook, we extended our exploratory corpus analysis by examining how the choice of **time-bin structure** affects diachronic interpretation. We compared a **fixed 500-year binning scheme**, which preserves equal chronological width, with a **quantile-based scheme**, which creates more balanced distributions of material across periods.

This comparison showed that time bins are not just a technical detail. They influence how we summarize the corpus, how evenly texts and tokens are distributed, and how distinctive lexical patterns appear in methods such as TF–IDF. A fixed-width scheme may be more historically intuitive, but it can also produce sparse or overloaded periods. A quantile scheme may improve balance, but it can blur historically meaningful temporal boundaries.

The broader lesson is that temporal analysis always involves a trade-off between **historical interpretability** and **statistical comparability**. Rather than assuming that one scheme reveals the “true” structure of the corpus, we should treat time bins as an analytical design choice whose consequences need to be inspected and justified.

```mermaid
flowchart TB
    A0["00<br/>Bootcamp"] --> P1

    subgraph P1["Part I — Corpus building and analysis"]
        direction LR
        A1a["01a<br/>Corpus metadata"] --> A1b["01b<br/>Corpus building"] --> A2["02<br/>Preprocessing"] --> A3["03<br/>Distributions + time"] --> A4a["04a<br/>Lexical exploration"] --> A4b["04b<br/>Embedding"]
    end

    subgraph P2["Part II — Linguistic annotations"]
        direction LR
        A5a["05a<br/>spaCy annotation"] --> A5b["05b<br/>Relation extraction"] --> A6a["06a<br/>NER"] --> A6b["06b<br/>Custom NER"]
    end

    subgraph P3["Part III — Representations"]
        direction LR
        A7["07<br/>BoW + TF-IDF"] --> A8a["08a<br/>Embeddings"] --> A8b["08b<br/>Transformers"]
    end

    subgraph P4["Part IV — Models and interpretation"]
        direction LR
        A9["09<br/>Classification"] --> A10["10<br/>Custom NER training"] --> A11["11<br/>Topic modeling"] --> A12["12<br/>Semantic shift"]
    end

    P1 --> P2
    P2 --> P3
    P3 --> P4

    classDef start fill:#f3f0ff,stroke:#6f42c1,stroke-width:1.5px,color:#111;
    classDef prep fill:#eef7ff,stroke:#1f77b4,stroke-width:1.5px,color:#111;
    classDef annot fill:#eefaf0,stroke:#2ca02c,stroke-width:1.5px,color:#111;
    classDef repr fill:#fff7e6,stroke:#ff8c00,stroke-width:1.5px,color:#111;
    classDef model fill:#fff0f0,stroke:#d62728,stroke-width:1.5px,color:#111;

    classDef highlight fill:#fff3b0,stroke:#f5a623,stroke-width:4px,color:#111;

    class A1a,A1b,A2,A3,A4a,A4b prep;
    class A5a,A5b,A6a,A6b annot;
    class A7,A8a,A8b repr;
    class A9,A10,A11,A12 model;

    class A7 highlight;
```